[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansensio/axr/blob/master/axr/00_intro.ipynb)

# Aprendizaje por Refuerzo

Empezamos creando la clase Board que representara el tablero de 4 en raya iniciando una matriz llena de ceros al igual que permitira ver si alguien gano, perdio o hubo empate en funcion a las diferentes maneras de ganar en 4 en raya y reinicia el tablero si se inicia una nueva partida 

In [1]:
import numpy as np

class Board():
    def __init__(self):
        self.state = np.zeros((4,4))  # Cambia a 4x4

    def valid_moves(self):
        return [(i, j) for j in range(4) for i in range(4) if self.state[i, j] == 0]

    def update(self, symbol, row, col):
        if self.state[row, col] == 0:
            self.state[row, col] = symbol
        else:
            raise ValueError ("movimiento ilegal !")

    def is_game_over(self):
        # Comprobar filas y columnas
        for i in range(4):
            if np.all(self.state[i, :] == 1) or np.all(self.state[:, i] == 1):
                return 1
            if np.all(self.state[i, :] == -1) or np.all(self.state[:, i] == -1):
                return -1
        # Comprobar diagonales principales
        if np.all([self.state[i, i] == 1 for i in range(4)]) or np.all([self.state[i, 3-i] == 1 for i in range(4)]):
            return 1
        if np.all([self.state[i, i] == -1 for i in range(4)]) or np.all([self.state[i, 3-i] == -1 for i in range(4)]):
            return -1
        # Empate
        if len(self.valid_moves()) == 0:
            return 0
        # Seguir jugando
        return None

    def reset(self):
        self.state = np.zeros((4,4))

# El Juego

Gesrionamos las partidas entre 2 agentes donde definimos las reglas del juego, recompensas y finalización del mismo 

In [2]:
from tqdm import tqdm

class Game():
    def __init__(self, player1, player2):
        player1.symbol = 1
        player2.symbol = -1
        self.players = [player1, player2]
        self.board = Board()

    def selfplay(self, rounds=100):
        wins = [0, 0]
        for i in tqdm(range(1, rounds + 1)):
            self.board.reset()
            for player in self.players:
                player.reset()
            game_over = False
            while not game_over:
                for player in self.players:
                    action = player.move(self.board)
                    self.board.update(player.symbol, action[0], action[1])
                    for player in self.players:
                        player.update(self.board)
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break
            self.reward()
            for ix, player in enumerate(self.players):
                if self.board.is_game_over() == player.symbol:
                    wins[ix] += 1
        return wins


    def reward(self):
        winner = self.board.is_game_over()
        if winner == 0: # empate
            for player in self.players:
                player.reward(0.5)
        else: # le damos 1 recompensa al jugador que gana
            for player in self.players:
                if winner == player.symbol:
                    player.reward(1)
                else:
                    player.reward(0)

# Agente

Creamos la Clase del Agente al cual lo entrenaremos para que aprenda a jugar con una tasa de aprendizaje, probabilidad de explorar y movimiento en 300000 partidas con la exploración y explotación

In [5]:
class Agent():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {} # tabla con pares estado -> valor
        self.alpha = alpha         # learning rate
        self.positions = []       # guardamos todas las posiciones de la partida
        self.prob_exp = prob_exp   # probabilidad de explorar

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        # exploracion
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]
        # explotacion
        max_value = -1000
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(4*4))  # <-- Cambiado a 4*4
            value = 0 if self.value_function.get(next_state) is None else self.value_function.get(next_state)
            if value >= max_value:
                max_value = value
                best_row, best_col = row, col
        return best_row, best_col

    def update(self, board):
        self.positions.append(str(board.state.reshape(4*4)))  # <-- Cambiado a 4*4

    def reward(self, reward):
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

# ...código existente...

agent1 = Agent(prob_exp=0.5)
agent2 = Agent()

game = Game(agent1, agent2)

game.selfplay(300000)

100%|██████████| 300000/300000 [42:04<00:00, 118.84it/s] 


[103983, 83634]

# Estados

Veremos los estados del tablero donde el Agente considero favorables y como aprendio a valorar las posiciones en el entrenamiento

In [6]:
import pandas as pd

funcion_de_valor = sorted(agent1.value_function.items(), key=lambda kv: kv[1], reverse=True)
tabla = pd.DataFrame({'estado': [x[0] for x in funcion_de_valor], 'valor': [x[1] for x in funcion_de_valor]})

tabla

,estado,valor
0,[ 1. 0. 0. 0. 1. 0. 0. -1. 1. 0. 0. -...,1.0
1,[ 0. 0. 1. 0. 0. 0. 1. -1. 0. 0. 1. -...,1.0
2,[-1. 0. 0. 1. 0. 0. -1. 1. -1. 0. 0. ...,1.0
3,[ 0. 0. 0. 1. -1. 0. -1. 1. 0. 0. 0. ...,1.0
4,[ 0. 0. 0. 1. 0. 0. -1. 1. -1. -1. 0. ...,1.0
...,...,...
1105881,[ 0. 0. 1. -1. 1. 1. 0. -1. 1. -1. -1. -...,0.0
1105882,[ 0. 0. 1. 0. 1. 1. 0. -1. 1. -1. -1. -...,0.0
1105883,[ 0. 0. 1. 0. 1. 1. 0. -1. 1. -1. -1. -...,0.0
1105884,[ 0. 0. 1. 0. 1. 1. 0. 0. 1. -1. -1. -...,0.0


# Evaluación del Agente

Hacemos una evaluación en 600 partidas del agente entrenado para ver cuantas victorias, derrotas y empeates tiene en porcentaje contra otro agente

In [34]:
import random

class RandomAgent:
    def __init__(self):
        self.symbol = None
    def reset(self):
        pass
    def move(self, board, explore=True):
        return random.choice(board.valid_moves())
    def update(self, board):
        pass
    def reward(self, reward):
        pass

def evaluar_agente(agente, partidas=1000):
    agente.symbol = 1
    oponente = RandomAgent()
    oponente.symbol = -1
    resultados = {1: 0, -1: 0, 0: 0}
    for _ in range(partidas):
        board = Board()
        turno = 1  # 1: agente, -1: oponente
        while True:
            if turno == 1:
                fila, col = agente.move(board, explore=False)
                board.update(1, fila, col)
            else:
                fila, col = oponente.move(board)
                board.update(-1, fila, col)
            resultado = board.is_game_over()
            if resultado is not None:
                resultados[resultado] += 1
                break
            turno *= -1
    print(f"De {partidas} partidas contra un oponente aleatorio:")
    print(f"Victorias del agente: {resultados[1]} ({resultados[1]/partidas*100:.2f}%)")
    print(f"Derrotas del agente: {resultados[-1]} ({resultados[-1]/partidas*100:.2f}%)")
    print(f"Empates: {resultados[0]} ({resultados[0]/partidas*100:.2f}%)")

evaluar_agente(agent2, partidas=600)

De 600 partidas contra un oponente aleatorio:
Victorias del agente: 166 (27.67%)
Derrotas del agente: 262 (43.67%)
Empates: 172 (28.67%)
